# pandas Basics
## MAD 2502 — Applied Data

**Big idea:** A DataFrame lets you inspect, filter, summarize, transform, and communicate tabular data.

This notebook is **standalone**. You do not need to complete another MAD 2502 notebook first.

By the end, you should be able to load a CSV, inspect a DataFrame, select and filter rows, handle missing values, make grouped summaries, create a graph, and save a result.

> **Lesson rhythm:** Understand → Predict → Run → Change → Use → Practice → Connect


[View on GitHub](https://github.com/mathguy-uf/MAD2502/blob/main/applied_data/02_pandas_basics.ipynb) · [Run in Google Colab](https://colab.research.google.com/github/mathguy-uf/MAD2502/blob/main/applied_data/02_pandas_basics.ipynb) · [Launch Binder/JupyterLab](https://mybinder.org/v2/gh/mathguy-uf/MAD2502/main?urlpath=lab/tree/applied_data/02_pandas_basics.ipynb)

Run cells from the top with **Shift+Enter**. Restart the kernel and run all to check your work. Each tutorial includes its own imports and data. Hosted sessions are temporary: download your work before leaving.

In [ ]:
# If an import fails, uncomment and run this once, then restart the kernel.
# %pip install pandas matplotlib requests

## 1. Understand

We will use a small fictional campus-events dataset. **One row represents one event.**

Before analyzing any dataset, ask:

- What does one row represent?
- What does each column mean?
- Which columns are numeric?
- Are values missing?

The code below first looks for the CSV in the repository. If it is not present — for example, when you open only this notebook in Google Colab — it downloads the same CSV directly from GitHub.


A CSV is a text file with comma-separated fields. A **DataFrame** is a table; a **Series** is one labeled column. `pd` is a short name for the pandas package. The loader below is setup: you may reuse it before understanding every line. Our workflow is **inspect → ask → select → compute → check → communicate**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from io import StringIO
from IPython.display import display

LOCAL_PATHS = [
    Path("applied_data/data/campus_events.csv"),
    Path("data/campus_events.csv"),
]
RAW_URL = (
    "https://raw.githubusercontent.com/mathguy-uf/"
    "MAD2502/main/applied_data/data/campus_events.csv"
)

data_path = next((path for path in LOCAL_PATHS if path.exists()), None)

if data_path is not None:
    events = pd.read_csv(data_path)
    print("Loaded local file:", data_path)
else:
    response = requests.get(RAW_URL, timeout=30)
    response.raise_for_status()
    events = pd.read_csv(StringIO(response.text))
    print("Loaded course data from GitHub.")

events.head()


### What just happened?

`pd.read_csv(...)` can read either a local file path or a web address that points directly to a CSV file.

That makes this example work in:

- a local clone of the repository;
- Binder/JupyterLab;
- Google Colab;
- most normal Jupyter environments with internet access.


## 2. Predict

Without running the next cell, predict what each expression tells you:

```python
events.head(3)
events.shape
events.shape[0]
events["category"]
events[["event_name", "attendance"]]
```

Which expression returns a **Series**? Which returns a **DataFrame**?


In [ ]:
print("shape:", events.shape)
print("columns:", list(events.columns))
print("category type:", type(events["category"]))

display(events.head(3))
display(events[["event_name", "attendance"]].head(3))


## 3. Run

Now use Boolean conditions to filter rows. Before looking at the output, predict whether each filter will keep many rows or only a few.


Start with a column (`events["attendance"]`), compare every value (`>= 100`), then use the resulting True/False values inside `events[...]` to select rows. A missing attendance will not satisfy the comparison.

In [ ]:
large_events = events[events["attendance"] >= 100]
display(large_events[["event_name", "attendance"]])

free_outdoor = events[
    (events["cost"] == 0)
    & (events["location_type"] == "Outdoor")
]
display(
    free_outdoor[
        ["event_name", "cost", "location_type"]
    ]
)


In pandas, `&` means **AND** and `|` means **OR** for element-by-element Boolean conditions. Put each comparison in parentheses.


## 4. Change

Make one controlled change at a time:

1. Change `attendance >= 100` to `attendance >= 150`. Predict whether the result gets longer or shorter.
2. Change AND (`&`) to OR (`|`) in the free/outdoor filter.
3. Change the category names below.
4. Sort attendance ascending instead of descending.


In [ ]:
music_or_sports = events[
    (events["category"] == "Music")
    | (events["category"] == "Sports")
]
display(music_or_sports)

display(
    events.sort_values(
        "attendance",
        ascending=False,
    )[["event_name", "attendance"]].head(5)
)


## 5. Use

Now answer a real question:

> **Which event categories tend to have larger attendance?**

First inspect missing values. Then build a grouped summary and a graph.


`groupby("category")` puts events with the same category together. `.agg(...)` calculates named summaries per group. `size` counts all rows; `count` counts known values; `mean` averages known values. Always check how many observations support a mean.

In [ ]:
print(events.isna().sum())


A missing value means **unknown**, not automatically zero.

For demonstration, we will fill missing ratings with the median of the known ratings. That is a modeling choice, not a universal rule.


Keep the original rating column. Compare missing counts before and after filling; imputation changes the data and can hide uncertainty.

In [ ]:
rating_median = events["rating"].median()

events["rating_filled"] = (
    events["rating"]
    .fillna(rating_median)
)

category_summary = (
    events.groupby("category", as_index=False)
    .agg(
        number_of_events=("event_name", "size"),
        known_attendance=("attendance", "count"),
        mean_attendance=("attendance", "mean"),
        median_cost=("cost", "median"),
    )
    .sort_values(
        "mean_attendance",
        ascending=False,
    )
)

display(category_summary)


In [ ]:
plot_data = category_summary.sort_values(
    "mean_attendance"
)

ax = plot_data.plot(
    x="category",
    y="mean_attendance",
    kind="barh",
    legend=False,
    figsize=(7, 4),
)

ax.set(
    xlabel="Mean attendance",
    ylabel="Event category",
    title="Mean attendance by event category",
)

plt.tight_layout()
plt.show()


### Check and communicate
`value_counts()` counts how often each category appears. The sum should equal the number of rows when missing categories are included. `sort_values()` rearranges rows; it does not change measurements. The next horizontal bar chart compares category means; read it alongside the known-attendance counts. These events are fictional, so the graph is practice, not a finding about UF.

In [ ]:
category_counts = events["category"].value_counts(dropna=False)
display(category_counts)
print("counts total:", category_counts.sum(), "rows:", len(events))

### Optional: save your result

Saving a cleaned or summarized table is useful in larger projects because one stage of the project can become the input to the next stage.


In [ ]:
output_path = Path("category_summary.csv")
category_summary.to_csv(
    output_path,
    index=False,
)
print("saved:", output_path.resolve())


## 6. Practice

Use `events` to answer each question.

1. Which event had the highest attendance?
2. How many events were free?
3. What was the median attendance for outdoor events?
4. Which category had the highest average rating?
5. Create a table containing only events with attendance above the overall median.
6. Use `value_counts()` on one categorical column.
7. Make one labeled graph that answers a question about the dataset.
8. Write one sentence explaining what a single row represents.


## 7. Connect

The basic pandas workflow is:

**inspect → ask → select → compute → check → communicate**

This pattern appears throughout the capstone.

Before using a complicated method, make sure you can answer:

- What does one row represent?
- What values are missing?
- What rows did your filters remove?
- What does your summary actually measure?
- Does your graph answer a question?

**Next connection:** The Statistics Villains notebook adds one new challenge — getting a table from a webpage before doing the pandas analysis.
